In [10]:
# ============================================================
# TÜRK MARKET SATIŞ VERİSİ — BİRLİKTELİK KURALI ANALİZİ
# M4 MacBook Pro (10 CPU, 16 GB RAM) için optimize edilmiştir
#
# AKIM:
#   Ham veri → Temizlik → Sepet matrisi → FP-Growth →
#   Association Rules → Filtreleme → Kayıt
# ============================================================
 
# ============================================================
# 1️⃣ KÜTÜPHANELER
# ============================================================
 
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import kagglehub
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
 
warnings.filterwarnings("ignore")
 
print("=" * 60)
print("Türk Market Satış — Birliktelik Kuralı Analizi")
print("=" * 60)
 
# ============================================================
# 2️⃣ VERİYİ İNDİR VE YÜKLE
# ============================================================
 
print("\n[1/6] Veri indiriliyor...")
 
dataset_path = kagglehub.dataset_download(
    "omercolakoglu/turkish-market-sales-dataset-with-9000items"
)
print(f"İndirilen klasör: {dataset_path}")
 
# Tüm Excel dosyalarını bul (alt klasörler dahil)
excel_files = []
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(('.xlsx', '.xls')):
            excel_files.append(os.path.join(root, file))
 
if not excel_files:
    raise FileNotFoundError(
        f"Klasörde Excel dosyası bulunamadı.\n"
        f"Klasör içeriği: {os.listdir(dataset_path)}"
    )
 
print(f"Bulunan Excel dosyaları: {len(excel_files)}")
for f in excel_files:
    print(f"  → {os.path.basename(f)}")
 
# Tüm Excel dosyalarını oku ve birleştir
dfs = []
for f in excel_files:
    print(f"\nOkunuyor: {os.path.basename(f)} ...")
    try:
        _df = pd.read_excel(f, usecols=lambda c: c in ['FICHENO', 'ITEMNAME', 'AMOUNT'])
        print(f"  Satır: {len(_df):,} | Sütunlar: {list(_df.columns)}")
        dfs.append(_df)
    except Exception as e:
        print(f"  ⚠️  Okunamadı: {e}")
 
if not dfs:
    raise ValueError("Hiçbir dosya okunamadı.")
 
df = pd.concat(dfs, ignore_index=True)
print(f"\nToplam ham satır: {len(df):,}")


Türk Market Satış — Birliktelik Kuralı Analizi

[1/6] Veri indiriliyor...
İndirilen klasör: /Users/semra/.cache/kagglehub/datasets/omercolakoglu/turkish-market-sales-dataset-with-9000items/versions/2
Bulunan Excel dosyaları: 1
  → MarketSales.xlsx

Okunuyor: MarketSales.xlsx ...
  Satır: 611,108 | Sütunlar: ['ITEMNAME', 'FICHENO', 'AMOUNT']

Toplam ham satır: 611,108


In [15]:
# ============================================================
# 3️⃣ VERİ TEMİZLİĞİ
# ============================================================
 
print("\n[2/7] Veri temizleniyor...")
 
df = df[['FICHENO', 'ITEMNAME', 'AMOUNT']].copy()
df = df.dropna(subset=['FICHENO', 'ITEMNAME'])
df = df[df['AMOUNT'] > 0]
df['ITEMNAME'] = df['ITEMNAME'].astype(str).str.strip().str.upper()
df['FICHENO']  = df['FICHENO'].astype(str).str.strip()
 
# Gürültü kaldır
noise_pattern = r"POSET|POŞET|KARGO|İNDİRİM|INDIRIM|TEST|İADE|IADE|HİZMET|HIZMET"
df = df[~df['ITEMNAME'].str.contains(noise_pattern, case=False, na=False, regex=True)]
 
# Nadir ürünleri kaldır — min 30 fiş
# (önceki 50'den 30'a düşürdük → daha fazla ürün dahil)
item_counts   = df['ITEMNAME'].value_counts()
popular_items = item_counts[item_counts >= 30].index
df = df[df['ITEMNAME'].isin(popular_items)]
 
# Tek ürünlü fişleri kaldır
basket_size   = df.groupby("FICHENO")["ITEMNAME"].nunique()
valid_baskets = basket_size[basket_size >= 2].index
df = df[df["FICHENO"].isin(valid_baskets)]
 
print(f"Temiz fiş sayısı  : {df['FICHENO'].nunique():,}")
print(f"Benzersiz ürün    : {df['ITEMNAME'].nunique():,}")
print(f"Toplam satır      : {len(df):,}")
 



[2/7] Veri temizleniyor...
Temiz fiş sayısı  : 82,128
Benzersiz ürün    : 2,088
Toplam satır      : 457,565


In [16]:
# ============================================================
# 4️⃣ SEPET MATRİSİ
# ============================================================
 
print("\n[3/7] Sepet matrisi oluşturuluyor...")
 
transactions = (
    df.groupby("FICHENO")["ITEMNAME"]
    .apply(list)
    .tolist()
)
 
te       = TransactionEncoder()
te_array = te.fit_transform(transactions)
basket   = pd.DataFrame(te_array, columns=te.columns_)
 
print(f"Sepet: {basket.shape[0]:,} fiş × {basket.shape[1]:,} ürün")
print(f"Bellek: {basket.memory_usage(deep=True).sum() / 1024**2:.1f} MB")



[3/7] Sepet matrisi oluşturuluyor...
Sepet: 82,128 fiş × 2,088 ürün
Bellek: 163.5 MB


In [17]:
# ============================================================
# 5️⃣ FP-GROWTH — DÜŞÜK SUPPORT
#
# Support grid düşürüldü: 0.01 → 0.003
# Bu sayede daha nadir ama anlamlı kombinasyonlar da yakalanır
# max_len=3 korundu — bellek dostu
# ============================================================
 
print("\n[4/7] FP-Growth çalışıyor (genişletilmiş)...")
 
# Düşük support → daha fazla kural
support_grid    = [0.003, 0.005, 0.007, 0.01]
confidence_grid = [0.3, 0.4, 0.5]
 
results    = []
all_rules  = []  # tüm kuralları biriktir (görselleştirme için)
start_time = time.time()
 
for support in support_grid:
    print(f"\n  Support = {support} ...")
    t0 = time.time()
 
    frequent_items = fpgrowth(
        basket,
        min_support=support,
        use_colnames=True,
        max_len=3
    )
 
    t1 = time.time()
 
    if frequent_items.empty:
        print(f"    → Sık öğe yok, atlanıyor.")
        continue
 
    print(f"    → {len(frequent_items):,} sık öğe ({t1-t0:.1f}s)")
 
    for conf in confidence_grid:
        rules = association_rules(
            frequent_items,
            metric="confidence",
            min_threshold=conf,
            num_itemsets=len(frequent_items)
        )
 
        if rules.empty:
            continue
 
        rules = rules[rules['lift'] > 1.0]
        rules = rules[rules['conviction'] > 1.0]
 
        if rules.empty:
            continue
 
        rules['support_param']    = support
        rules['confidence_param'] = conf
        all_rules.append(rules)
 
        zhang = rules['zhangs_metric'].mean() if 'zhangs_metric' in rules.columns else 0
 
        results.append({
            'support'       : support,
            'confidence'    : conf,
            'n_rules'       : len(rules),
            'avg_lift'      : round(rules['lift'].mean(), 3),
            'avg_confidence': round(rules['confidence'].mean(), 3),
            'zhang'         : round(zhang, 3),
            'rules_df'      : rules
        })
 
        print(f"    Conf={conf} → {len(rules):,} kural | "
              f"Lift={rules['lift'].mean():.2f} | Zhang={zhang:.2f}")
 
print(f"\nToplam süre: {time.time() - start_time:.1f}s")
 
if not results:
    raise ValueError("Hiç kural üretilemedi. Support eşiğini daha da düşür.")
 



[4/7] FP-Growth çalışıyor (genişletilmiş)...

  Support = 0.003 ...
    → 512 sık öğe (13.6s)
    Conf=0.3 → 31 kural | Lift=8.48 | Zhang=0.89
    Conf=0.4 → 12 kural | Lift=9.03 | Zhang=0.90
    Conf=0.5 → 1 kural | Lift=12.07 | Zhang=0.93

  Support = 0.005 ...
    → 236 sık öğe (5.5s)
    Conf=0.3 → 11 kural | Lift=8.42 | Zhang=0.89
    Conf=0.4 → 6 kural | Lift=8.17 | Zhang=0.89
    Conf=0.5 → 1 kural | Lift=12.07 | Zhang=0.93

  Support = 0.007 ...
    → 133 sık öğe (3.5s)
    Conf=0.3 → 6 kural | Lift=7.33 | Zhang=0.88
    Conf=0.4 → 4 kural | Lift=7.32 | Zhang=0.88

  Support = 0.01 ...
    → 86 sık öğe (2.2s)
    Conf=0.3 → 5 kural | Lift=6.98 | Zhang=0.88
    Conf=0.4 → 4 kural | Lift=7.32 | Zhang=0.88

Toplam süre: 24.8s


In [18]:
# ============================================================
# 6️⃣ EN İYİ PARAMETRELERİ SEÇ VE KURALLARI BİRLEŞTİR
# ============================================================
 
print("\n[5/7] Sonuçlar derleniyor...")
 
results_summary = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'rules_df'}
    for r in results
])
print("\nTüm kombinasyon sonuçları:")
print(results_summary.to_string(index=False))
 
# En iyi parametreler (zhang + lift)
best_result = max(results, key=lambda r: (r['zhang'], r['avg_lift']))
best_rules  = best_result['rules_df'].copy()
best_params = (best_result['support'], best_result['confidence'])
 
# Tüm kuralları birleştir (görselleştirme için geniş set kullan)
all_rules_df = pd.concat(all_rules, ignore_index=True)
# Duplikat kuralları kaldır (aynı antecedent-consequent çifti)
all_rules_df = all_rules_df.drop_duplicates(
    subset=['antecedents', 'consequents']
)
all_rules_df = all_rules_df.sort_values('lift', ascending=False)
 
print(f"\nEn iyi parametreler : support={best_params[0]}, confidence={best_params[1]}")
print(f"En iyi kural sayısı : {len(best_rules):,}")
print(f"Toplam benzersiz kural (tüm parametreler): {len(all_rules_df):,}")
 
# Okunabilir format
for col in ['antecedents', 'consequents']:
    all_rules_df[col] = all_rules_df[col].apply(
        lambda x: ', '.join(sorted(list(x)))
    )
    best_rules[col] = best_rules[col].apply(
        lambda x: ', '.join(sorted(list(x)))
    )
 
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'conviction']
if 'zhangs_metric' in best_rules.columns:
    display_cols.append('zhangs_metric')
 
print("\nEn güçlü 20 kural:")
print("=" * 80)
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 140)
print(all_rules_df[display_cols].head(20).to_string(index=False))
 
# CSV kaydet
all_rules_df[display_cols].to_csv(
    "results/association_rules_full.csv", index=False, encoding='utf-8-sig'
)
print("\nTüm kurallar → results/association_rules_full.csv")



[5/7] Sonuçlar derleniyor...

Tüm kombinasyon sonuçları:
 support  confidence  n_rules  avg_lift  avg_confidence  zhang
   0.003         0.3       31     8.478           0.389  0.886
   0.003         0.4       12     9.026           0.446  0.896
   0.003         0.5        1    12.069           0.511  0.929
   0.005         0.3       11     8.425           0.404  0.891
   0.005         0.4        6     8.168           0.451  0.887
   0.005         0.5        1    12.069           0.511  0.929
   0.007         0.3        6     7.328           0.410  0.882
   0.007         0.4        4     7.317           0.433  0.879
   0.010         0.3        5     6.982           0.415  0.876
   0.010         0.4        4     7.317           0.433  0.879

En iyi parametreler : support=0.003, confidence=0.5
En iyi kural sayısı : 1
Toplam benzersiz kural (tüm parametreler): 31

En güçlü 20 kural:
          antecedents         consequents  support  confidence      lift  conviction  zhangs_metric
      

In [21]:
# ============================================================
# 7️⃣ GÖRSELLEŞTİRME
# ============================================================
import matplotlib
matplotlib.use('Agg')  # GUI olmadan kaydet
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
print("\n[6/7] Görselleştirmeler oluşturuluyor...")
 
# Görselleştirme için en iyi N kural
TOP_N    = min(30, len(all_rules_df))
top_rules = all_rules_df.head(TOP_N).copy()
 
# ── RENK PALETİ ──────────────────────────────────────────────
COLORS = {
    'bg'      : '#0f1117',
    'panel'   : '#1a1d27',
    'accent1' : '#7c6af7',   # mor — antecedent
    'accent2' : '#f97316',   # turuncu — consequent
    'edge'    : '#4ade80',   # yeşil — kural oku
    'text'    : '#e2e8f0',
    'subtext' : '#94a3b8',
    'grid'    : '#2d3148',
}
 
plt.rcParams.update({
    'figure.facecolor' : COLORS['bg'],
    'axes.facecolor'   : COLORS['panel'],
    'axes.edgecolor'   : COLORS['grid'],
    'text.color'       : COLORS['text'],
    'axes.labelcolor'  : COLORS['text'],
    'xtick.color'      : COLORS['subtext'],
    'ytick.color'      : COLORS['subtext'],
    'grid.color'       : COLORS['grid'],
    'grid.linewidth'   : 0.5,
    'font.family'      : 'DejaVu Sans',
})
 
# ── PLOT 1: SCATTER — Support vs Confidence (lift ile renk) ──
 
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(COLORS['bg'])
 
scatter = ax.scatter(
    all_rules_df['support'],
    all_rules_df['confidence'],
    c=all_rules_df['lift'],
    cmap='plasma',
    s=all_rules_df['lift'] * 12,
    alpha=0.75,
    edgecolors='white',
    linewidths=0.3,
    zorder=3
)
 
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Lift Değeri', color=COLORS['text'], fontsize=11)
cbar.ax.yaxis.set_tick_params(color=COLORS['subtext'])
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=COLORS['subtext'])
 
# En iyi 5 kuralı etiketle
for _, row in all_rules_df.head(5).iterrows():
    ax.annotate(
        f"{row['antecedents'][:15]}→{row['consequents'][:12]}",
        xy=(row['support'], row['confidence']),
        xytext=(8, 6),
        textcoords='offset points',
        fontsize=7,
        color=COLORS['text'],
        alpha=0.9,
        bbox=dict(boxstyle='round,pad=0.2', facecolor=COLORS['panel'],
                  edgecolor=COLORS['accent1'], alpha=0.8)
    )
 
ax.set_xlabel('Support', fontsize=12, labelpad=8)
ax.set_ylabel('Confidence', fontsize=12, labelpad=8)
ax.set_title('Birliktelik Kuralları — Support vs Confidence\n(Daire büyüklüğü ve rengi = Lift)',
             fontsize=13, pad=15, color=COLORS['text'])
ax.grid(True, alpha=0.3, zorder=0)
 
plt.tight_layout()
plt.savefig('results/plots/01_scatter_support_confidence.png', dpi=150,
            bbox_inches='tight', facecolor=COLORS['bg'])
plt.close()
print("  ✓ Plot 1: Scatter kaydedildi")
 
# ── PLOT 2: BAR — En yüksek lift'e sahip kurallar ────────────
 
top20 = all_rules_df.head(20).copy()
top20['kural'] = top20['antecedents'].str[:20] + ' → ' + top20['consequents'].str[:15]
 
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor(COLORS['bg'])
 
bars = ax.barh(
    range(len(top20)),
    top20['lift'],
    color=[COLORS['accent1'] if i % 2 == 0 else COLORS['accent2']
           for i in range(len(top20))],
    edgecolor=COLORS['bg'],
    linewidth=0.5,
    alpha=0.9
)
 
# Değer etiketleri
for i, (bar, val) in enumerate(zip(bars, top20['lift'])):
    ax.text(val + 0.1, i, f'{val:.2f}',
            va='center', fontsize=8.5, color=COLORS['text'])
 
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['kural'], fontsize=9)
ax.set_xlabel('Lift Değeri', fontsize=12, labelpad=8)
ax.set_title('En Güçlü 20 Birliktelik Kuralı (Lift\'e Göre)',
             fontsize=13, pad=15, color=COLORS['text'])
ax.axvline(x=1, color=COLORS['subtext'], linestyle='--',
           alpha=0.5, linewidth=1, label='Lift = 1 (bağımsız)')
ax.legend(fontsize=9, facecolor=COLORS['panel'], edgecolor=COLORS['grid'])
ax.grid(True, axis='x', alpha=0.3)
ax.invert_yaxis()
 
plt.tight_layout()
plt.savefig('results/plots/02_bar_top_rules.png', dpi=150,
            bbox_inches='tight', facecolor=COLORS['bg'])
plt.close()
print("  ✓ Plot 2: Bar chart kaydedildi")
 
# ── PLOT 3: NETWORK GRAPH ─────────────────────────────────────
 
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor(COLORS['bg'])
ax.set_facecolor(COLORS['bg'])
 
G     = nx.DiGraph()
edges = []
 
for _, row in top_rules.iterrows():
    ant = row['antecedents']
    con = row['consequents']
    G.add_edge(ant, con, weight=row['lift'], confidence=row['confidence'])
    edges.append((ant, con, row['lift']))
 
# Düğüm renkleri: antecedent vs consequent
antecedents_set  = set(top_rules['antecedents'])
consequents_set  = set(top_rules['consequents'])
node_colors = []
node_sizes  = []
 
for node in G.nodes():
    degree = G.degree(node)
    node_sizes.append(300 + degree * 200)
    if node in antecedents_set and node in consequents_set:
        node_colors.append('#f59e0b')   # sarı — her ikisi de
    elif node in antecedents_set:
        node_colors.append(COLORS['accent1'])   # mor — sadece antecedent
    else:
        node_colors.append(COLORS['accent2'])   # turuncu — sadece consequent
 
# Layout
pos = nx.spring_layout(G, k=2.5, seed=42, iterations=50)
 
# Kenar kalınlıkları
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_widths = [0.5 + (w / max_w) * 3 for w in edge_weights]
 
# Kenarlar
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color=COLORS['edge'],
    width=edge_widths,
    alpha=0.6,
    arrows=True,
    arrowsize=15,
    arrowstyle='-|>',
    connectionstyle='arc3,rad=0.1',
    min_source_margin=20,
    min_target_margin=20
)
 
# Düğümler
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.92,
    linewidths=1.5,
    edgecolors='white'
)
 
# Etiketler
nx.draw_networkx_labels(
    G, pos, ax=ax,
    font_size=8,
    font_color=COLORS['bg'],
    font_weight='bold'
)
 
# Legend
legend_elements = [
    mpatches.Patch(color=COLORS['accent1'], label='Antecedent (neden)'),
    mpatches.Patch(color=COLORS['accent2'], label='Consequent (sonuç)'),
    mpatches.Patch(color='#f59e0b',         label='Her ikisi'),
    mpatches.Patch(color=COLORS['edge'],    label='Kural (ok kalınlığı = lift)'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9,
          facecolor=COLORS['panel'], edgecolor=COLORS['grid'],
          labelcolor=COLORS['text'])
 
ax.set_title(f'Birliktelik Kuralları Network Grafiği (Top {TOP_N} Kural)',
             fontsize=14, pad=15, color=COLORS['text'])
ax.axis('off')
 
plt.tight_layout()
plt.savefig('results/plots/03_network_graph.png', dpi=150,
            bbox_inches='tight', facecolor=COLORS['bg'])
plt.close()
print("  ✓ Plot 3: Network graph kaydedildi")
 
# ── PLOT 4: HEATMAP — Lift matrisi (en sık geçen ürünler) ────
 
print("  Heatmap oluşturuluyor...")
 
# En çok kuralda geçen ürünleri bul
all_items = (
    list(all_rules_df['antecedents']) +
    list(all_rules_df['consequents'])
)
top_items = pd.Series(all_items).value_counts().head(15).index.tolist()
 
# Lift matrisi oluştur
lift_matrix = pd.DataFrame(np.nan, index=top_items, columns=top_items)
 
for _, row in all_rules_df.iterrows():
    ant = row['antecedents']
    con = row['consequents']
    if ant in top_items and con in top_items:
        lift_matrix.loc[ant, con] = row['lift']
 
fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor(COLORS['bg'])
 
# NaN → 0 (görsel için)
heatmap_data = lift_matrix.fillna(0)
 
im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto', vmin=0)
 
ax.set_xticks(range(len(top_items)))
ax.set_yticks(range(len(top_items)))
ax.set_xticklabels(top_items, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(top_items, fontsize=9)
 
# Hücre değerleri
for i in range(len(top_items)):
    for j in range(len(top_items)):
        val = heatmap_data.values[i, j]
        if val > 0:
            ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                    fontsize=7.5,
                    color='black' if val > heatmap_data.values.max() * 0.5 else 'white',
                    fontweight='bold')
 
cbar2 = plt.colorbar(im, ax=ax)
cbar2.set_label('Lift Değeri', color=COLORS['text'], fontsize=11)
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=COLORS['subtext'])
 
ax.set_title('Ürünler Arası Lift Isı Haritası\n(Boş hücre = kural yok)',
             fontsize=13, pad=15, color=COLORS['text'])
 
plt.tight_layout()
plt.savefig('results/plots/04_heatmap_lift.png', dpi=150,
            bbox_inches='tight', facecolor=COLORS['bg'])
plt.close()
print("  ✓ Plot 4: Heatmap kaydedildi")
 
# ============================================================
# 8️⃣ ÖZET
# ============================================================
 
print("\n[7/7] Tamamlandı!")
print("\n" + "=" * 60)
print("ÖZET:")
print("=" * 60)
print(f"Toplam fiş          : {df['FICHENO'].nunique():,}")
print(f"Benzersiz ürün      : {df['ITEMNAME'].nunique():,}")
print(f"Toplam kural        : {len(all_rules_df):,}")
print(f"En iyi support      : {best_params[0]}")
print(f"En iyi confidence   : {best_params[1]}")
print(f"En yüksek lift      : {all_rules_df['lift'].max():.3f}")
print(f"Ortalama lift       : {all_rules_df['lift'].mean():.3f}")
print()
print("Kaydedilen dosyalar:")
print("  results/association_rules_full.csv")
print("  results/plots/01_scatter_support_confidence.png")
print("  results/plots/02_bar_top_rules.png")
print("  results/plots/03_network_graph.png")
print("  results/plots/04_heatmap_lift.png")
print("=" * 60)
print("✅ Analiz tamamlandı!")



[6/7] Görselleştirmeler oluşturuluyor...
  ✓ Plot 1: Scatter kaydedildi
  ✓ Plot 2: Bar chart kaydedildi
  ✓ Plot 3: Network graph kaydedildi
  Heatmap oluşturuluyor...
  ✓ Plot 4: Heatmap kaydedildi

[7/7] Tamamlandı!

ÖZET:
Toplam fiş          : 82,128
Benzersiz ürün      : 2,088
Toplam kural        : 31
En iyi support      : 0.003
En iyi confidence   : 0.5
En yüksek lift      : 15.020
Ortalama lift       : 8.478

Kaydedilen dosyalar:
  results/association_rules_full.csv
  results/plots/01_scatter_support_confidence.png
  results/plots/02_bar_top_rules.png
  results/plots/03_network_graph.png
  results/plots/04_heatmap_lift.png
✅ Analiz tamamlandı!
